# Face Verification with a Siamese Network (TensorFlow)

One-shot face verification: given two face images, predict whether they show the **same person**.
A shared CNN maps each image to an embedding, an L1-distance layer compares the two embeddings,
and a sigmoid classifier outputs a similarity score.

用 Siamese Network 做人臉驗證：輸入兩張臉，判斷是不是同一個人。

**Pipeline**

1. Setup & config
2. (Optional) Collect webcam images / migrate old data
3. Leak-free train / val / test split
4. Build pairs (self pairs + LFW pairs)
5. Model: embedding CNN + L1 distance + classifier
6. Custom training loop (`tf.GradientTape`) with early stopping
7. Evaluation on a held-out test set (threshold chosen on val)
8. Save / reload model
9. Verification demo (webcam in Colab)

Based on Koch et al., *Siamese Neural Networks for One-shot Image Recognition* (2015).

## 1. Setup

In [ ]:
import os, glob, json, random, shutil, uuid

import numpy as np
import tensorflow as tf
import matplotlib.pyplot as plt
from tensorflow.keras import layers, Model
from sklearn.metrics import roc_auc_score, roc_curve, confusion_matrix

try:
    from google.colab import drive
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    drive.mount('/content/drive')

# Avoid grabbing all GPU memory at once
for gpu in tf.config.list_physical_devices('GPU'):
    tf.config.experimental.set_memory_growth(gpu, True)

print('TensorFlow', tf.__version__, '| GPU:', tf.config.list_physical_devices('GPU'))

### Config
All paths and hyper-parameters live here. 所有路徑和超參數都集中在這裡。

LFW identity is parsed from the file name (`Person_Name_0001.jpg`), so LFW images can sit either in
the original `<person>/<img>.jpg` folders or flattened into one folder.

In [ ]:
PROJECT_DIR = os.environ.get('PROJECT_DIR', '/content/drive/MyDrive/Vita_face_recognition')

RAW_DIR = os.path.join(PROJECT_DIR, 'data', 'raw')   # original webcam captures only (no augmented copies)
ANC_DIR = os.path.join(RAW_DIR, 'anchor')
POS_DIR = os.path.join(RAW_DIR, 'positive')
LFW_DIR = os.environ.get('LFW_DIR', os.path.join(PROJECT_DIR, 'data', 'negative'))
OUTPUT_DIR = os.path.join(PROJECT_DIR, 'outputs')

IMG_SIZE = 100
BATCH_SIZE = 16
EPOCHS = int(os.environ.get('EPOCHS', 50))
LEARNING_RATE = 1e-4
PATIENCE = 8                       # early stopping on val loss
SPLIT_RATIOS = (0.70, 0.15, 0.15)  # train / val / test
PAIRS_PER_TYPE = {                 # number of pairs for EACH of the 4 pair types
    'train': int(os.environ.get('TRAIN_PAIRS', 3000)),
    'val':   int(os.environ.get('EVAL_PAIRS', 500)),
    'test':  int(os.environ.get('EVAL_PAIRS', 500)),
}
SEED = 42

random.seed(SEED); np.random.seed(SEED); tf.random.set_seed(SEED)
for d in (ANC_DIR, POS_DIR, OUTPUT_DIR):
    os.makedirs(d, exist_ok=True)

## 2. Data collection (optional)

### 2.1 Migrate data from the first version
The first version wrote augmented copies back into the same `anchor/` and `positive/` folders.
Original captures were saved with `uuid4` names, augmented copies with `uuid1` names,
so we can recover the originals by checking the UUID version.

第一版把增強後的圖片和原圖存在同一個資料夾。原圖檔名是 uuid4、增強圖是 uuid1，這裡只把原圖複製到 `data/raw/`。

In [ ]:
OLD_ANC_DIR = os.path.join(PROJECT_DIR, 'data', 'anchor')
OLD_POS_DIR = os.path.join(PROJECT_DIR, 'data', 'positive')
IMG_EXTS = ('.jpg', '.jpeg', '.png')

def is_original_capture(path):
    stem = os.path.splitext(os.path.basename(path))[0]
    try:
        return uuid.UUID(stem).version == 4
    except ValueError:
        return False

def migrate(old_dir, new_dir):
    if not os.path.isdir(old_dir) or os.listdir(new_dir):
        return 0
    files = [f for f in os.listdir(old_dir) if f.lower().endswith(IMG_EXTS)]
    kept = [f for f in files if is_original_capture(f)]
    for f in kept:
        shutil.copy2(os.path.join(old_dir, f), os.path.join(new_dir, f))  # copy2 keeps capture time
    print(f'{old_dir}: kept {len(kept)} originals, skipped {len(files) - len(kept)} augmented copies')
    return len(kept)

migrate(OLD_ANC_DIR, ANC_DIR)
migrate(OLD_POS_DIR, POS_DIR)
print('anchor:', len(os.listdir(ANC_DIR)), '| positive:', len(os.listdir(POS_DIR)))

### 2.2 Webcam collector (Colab only)
Captures a 250×250 center crop once per second into `data/raw/anchor` or `data/raw/positive`.
**Tip:** record several sessions (different lighting, backgrounds, days). The split below is
chronological, so later sessions become the val/test set.

在 Colab 用 webcam 自動拍照，每秒一張。建議分不同時段、不同光線拍，後面拍的會被分到測試集。

In [ ]:
COLLECT = False  # set True to open the collector

if IN_COLAB and COLLECT:
    from google.colab import output
    from IPython.display import HTML, Javascript, display
    from base64 import b64decode

    def save_image_callback(data_url, img_type):
        folder = {'anchor': ANC_DIR, 'positive': POS_DIR}.get(img_type)
        if folder is None:
            return
        with open(os.path.join(folder, f'{uuid.uuid4()}.jpg'), 'wb') as f:
            f.write(b64decode(data_url.split(',')[1]))
        return Javascript(f"updateCounter('{img_type}')")

    output.register_callback('notebook.save_image', save_image_callback)

    display(HTML('''
    <div style="font-family:sans-serif;text-align:center">
      <video id="video" width="400" height="300" autoplay style="border:3px solid #444;border-radius:8px"></video>
      <canvas id="canvas" style="display:none"></canvas><br>
      <button id="b-anchor">Auto Anchor</button>
      <button id="b-positive">Auto Positive</button>
      <button id="b-stop">Stop</button>
      <button id="b-close">Close camera</button>
      <p id="status">Ready</p>
      <p>anchor: <b id="count-anchor">0</b> | positive: <b id="count-positive">0</b></p>
    </div>
    <script>
    (async () => {
      const video = document.getElementById('video'), canvas = document.getElementById('canvas');
      const ctx = canvas.getContext('2d'), status = document.getElementById('status');
      const SIZE = 250; let timer = null; const counts = {anchor: 0, positive: 0};
      const stream = await navigator.mediaDevices.getUserMedia({video: {width: 640, height: 480}});
      video.srcObject = stream;
      await new Promise(r => video.onloadedmetadata = r);

      function capture(type) {
        const s = Math.min(video.videoWidth, video.videoHeight);
        canvas.width = SIZE; canvas.height = SIZE;
        ctx.drawImage(video, (video.videoWidth - s) / 2, (video.videoHeight - s) / 2, s, s, 0, 0, SIZE, SIZE);
        google.colab.kernel.invokeFunction('notebook.save_image', [canvas.toDataURL('image/jpeg', 0.9), type], {});
      }
      function stop() { clearInterval(timer); timer = null; status.innerText = 'Paused'; }
      function start(type) { stop(); status.innerText = 'Capturing ' + type + ' (1 / sec)'; timer = setInterval(() => capture(type), 1000); }
      window.updateCounter = t => { counts[t]++; document.getElementById('count-' + t).innerText = counts[t]; };
      document.getElementById('b-anchor').onclick = () => start('anchor');
      document.getElementById('b-positive').onclick = () => start('positive');
      document.getElementById('b-stop').onclick = stop;
      document.getElementById('b-close').onclick = () => { stop(); stream.getTracks().forEach(t => t.stop()); status.innerText = 'Camera closed'; };
    })();
    </script>
    '''))

## 3. Leak-free split

The first version reported ~96% val accuracy, but that number was inflated by **data leakage**:

1. `data.shuffle()` reshuffles every epoch *before* `take()/skip()`, so the train/test split changed each epoch and the model eventually saw every test pair.
2. Augmented copies were written to disk before splitting, so near-identical images ended up in both train and test.

Fixes in this version:

* Split **file paths** once, before anything else.
* Self images: **chronological** split (webcam frames 1 s apart are near-duplicates; random splitting would leak).
* LFW images: split **by identity**, so test faces belong to people never seen in training.
* Augmentation is applied **on the fly to the training set only**.

第一版 96% 的準確率因為資料洩漏被高估了。這版先切檔案路徑、自己的照片依拍攝時間切、LFW 依「人」切，資料增強只在訓練時即時做。

In [ ]:
def list_images(folder):
    return sorted(p for p in glob.glob(os.path.join(folder, '**', '*'), recursive=True)
                  if p.lower().endswith(IMG_EXTS))

def ratio_cut(n, ratios):
    a = int(n * ratios[0]); b = int(n * (ratios[0] + ratios[1]))
    return a, b

def chrono_split(paths, ratios):
    paths = sorted(paths, key=os.path.getmtime)
    a, b = ratio_cut(len(paths), ratios)
    return {'train': paths[:a], 'val': paths[a:b], 'test': paths[b:]}

def lfw_identity(path):
    # 'George_W_Bush_0001.jpg' -> 'George_W_Bush'
    return os.path.splitext(os.path.basename(path))[0].rsplit('_', 1)[0]

def identity_split(paths, ratios, seed):
    by_id = {}
    for p in paths:
        by_id.setdefault(lfw_identity(p), []).append(p)
    ids = sorted(by_id)
    random.Random(seed).shuffle(ids)
    a, b = ratio_cut(len(ids), ratios)
    groups = {'train': ids[:a], 'val': ids[a:b], 'test': ids[b:]}
    return {s: {i: by_id[i] for i in g} for s, g in groups.items()}

anc_split = chrono_split(list_images(ANC_DIR), SPLIT_RATIOS)
pos_split = chrono_split(list_images(POS_DIR), SPLIT_RATIOS)
lfw_split = identity_split(list_images(LFW_DIR), SPLIT_RATIOS, SEED)

for s in ('train', 'val', 'test'):
    n_people = len(lfw_split[s]); n_lfw = sum(len(v) for v in lfw_split[s].values())
    print(f'{s:5s} | anchor {len(anc_split[s]):5d} | positive {len(pos_split[s]):5d} | LFW {n_lfw:6d} imgs / {n_people:5d} people')

# Sanity checks: no file or LFW identity appears in two splits
for a, b in [('train', 'val'), ('train', 'test'), ('val', 'test')]:
    assert not set(anc_split[a]) & set(anc_split[b])
    assert not set(pos_split[a]) & set(pos_split[b])
    assert not set(lfw_split[a]) & set(lfw_split[b])
print('No overlap between splits ✔')

## 4. Build pairs

Four pair types, balanced:

| type | image A | image B | label |
|---|---|---|---|
| `self_pos` | my anchor | my positive | 1 |
| `self_neg` | my anchor | LFW face | 0 |
| `lfw_pos` | LFW person X | another photo of X | 1 |
| `lfw_neg` | LFW person X | LFW person Y | 0 |

The first version only had `self_pos` and `self_neg`. Because *all* positives were webcam photos and *all* negatives were LFW photos,
the model could get high accuracy just by telling **webcam vs. LFW** (lighting, background, image quality) instead of comparing faces.
Adding LFW-vs-LFW pairs removes that shortcut, and the `lfw_*` test score shows how well the model generalises to unseen people.

第一版的正樣本全部是 webcam、負樣本全部是 LFW，模型可能只學會分辨照片來源。加入 LFW 對 LFW 的配對可以避免這個捷徑。

In [ ]:
PAIR_TYPES = ['self_pos', 'self_neg', 'lfw_pos', 'lfw_neg']

def make_pairs(anchors, positives, lfw_by_id, n_per_type, seed):
    rng = random.Random(seed)
    lfw_all = [p for ps in lfw_by_id.values() for p in ps]
    multi = [i for i, ps in lfw_by_id.items() if len(ps) >= 2]   # people with >= 2 photos
    ids = list(lfw_by_id)
    pairs = []  # (path_a, path_b, label, type_idx)
    for _ in range(n_per_type):
        pairs.append((rng.choice(anchors), rng.choice(positives), 1, 0))
        pairs.append((rng.choice(anchors), rng.choice(lfw_all), 0, 1))
        i = rng.choice(multi); a, b = rng.sample(lfw_by_id[i], 2)
        pairs.append((a, b, 1, 2))
        i, j = rng.sample(ids, 2)
        pairs.append((rng.choice(lfw_by_id[i]), rng.choice(lfw_by_id[j]), 0, 3))
    return pairs

pairs = {s: make_pairs(anc_split[s], pos_split[s], lfw_split[s], PAIRS_PER_TYPE[s], SEED + k)
         for k, s in enumerate(['train', 'val', 'test'])}
{s: len(p) for s, p in pairs.items()}

### 4.1 Load images into memory
Each unique image is decoded and resized to 100×100 once, then stored as `uint8`
(about 30 KB per image). Pairs only hold indices into this array.

每張圖只讀取一次，存成 uint8 陣列放在記憶體，配對只存索引，比每個 epoch 從 Google Drive 讀檔快很多。

In [ ]:
AUTOTUNE = tf.data.AUTOTUNE

def load_image(path):
    img = tf.io.decode_image(tf.io.read_file(path), channels=3, expand_animations=False)
    img = tf.image.resize(img, (IMG_SIZE, IMG_SIZE))
    return tf.cast(tf.round(img), tf.uint8)

unique_paths = sorted({p for s in pairs for a, b, _, _ in pairs[s] for p in (a, b)})
path_to_idx = {p: i for i, p in enumerate(unique_paths)}

ds = tf.data.Dataset.from_tensor_slices(unique_paths).map(load_image, num_parallel_calls=AUTOTUNE).batch(256)
IMAGES = np.concatenate([b.numpy() for b in ds])
print('Loaded', IMAGES.shape, f'{IMAGES.nbytes / 1e6:.0f} MB')

def to_arrays(pair_list):
    a = np.array([path_to_idx[p[0]] for p in pair_list], dtype=np.int32)
    b = np.array([path_to_idx[p[1]] for p in pair_list], dtype=np.int32)
    y = np.array([p[2] for p in pair_list], dtype=np.float32)
    t = np.array([p[3] for p in pair_list], dtype=np.int32)
    return a, b, y, t

arrays = {s: to_arrays(p) for s, p in pairs.items()}

### 4.2 `tf.data` pipeline with on-the-fly augmentation (train only)

In [ ]:
IMAGES_T = tf.constant(IMAGES)

def augment(img):
    img = tf.image.random_flip_left_right(img)
    img = tf.image.random_brightness(img, 0.15)
    img = tf.image.random_contrast(img, 0.7, 1.2)
    img = tf.image.random_saturation(img, 0.8, 1.2)
    return tf.clip_by_value(img, 0.0, 1.0)

def make_dataset(split, training):
    a, b, y, _ = arrays[split]
    ds = tf.data.Dataset.from_tensor_slices((a, b, y))
    if training:
        ds = ds.shuffle(len(y), seed=SEED, reshuffle_each_iteration=True)  # shuffles WITHIN train only

    def fetch(ia, ib, label):
        xa = tf.cast(tf.gather(IMAGES_T, ia), tf.float32) / 255.0
        xb = tf.cast(tf.gather(IMAGES_T, ib), tf.float32) / 255.0
        if training:
            xa, xb = augment(xa), augment(xb)
        return (xa, xb), label

    return ds.map(fetch, num_parallel_calls=AUTOTUNE).batch(BATCH_SIZE).prefetch(AUTOTUNE)

train_ds = make_dataset('train', training=True)
val_ds = make_dataset('val', training=False)
test_ds = make_dataset('test', training=False)

In [ ]:
# Peek at a few training pairs
(xa, xb), yb = next(iter(train_ds))
fig, axes = plt.subplots(2, 6, figsize=(14, 5))
for k in range(6):
    axes[0, k].imshow(xa[k]); axes[1, k].imshow(xb[k])
    axes[0, k].set_title(f'label={int(yb[k])}')
    axes[0, k].axis('off'); axes[1, k].axis('off')
plt.tight_layout(); plt.show()

## 5. Model

* **Embedding CNN** (shared weights): 3 conv blocks with BatchNorm → GlobalAveragePooling → Dropout → Dense(1024).
* **L1 distance layer**: `|emb(A) - emb(B)|`.
* **Classifier**: Dense(1, sigmoid) → similarity score in [0, 1].

v1 used `Flatten → Dense(4096)` (~39M parameters) and collapsed to predicting 0 for everything.
Replacing it with GlobalAveragePooling + BatchNorm cut the model to ~1.2M parameters and made training stable.

第一版用 Flatten + Dense(4096)，約 3,900 萬參數，訓練崩潰成全部預測 0。改用 BatchNorm + GlobalAveragePooling 後只剩約 120 萬參數，訓練也穩定了。

In [ ]:
def make_embedding():
    inp = layers.Input(shape=(IMG_SIZE, IMG_SIZE, 3), name='input_image')
    x = layers.Conv2D(64, 10, activation='relu')(inp)
    x = layers.BatchNormalization()(x)
    x = layers.MaxPooling2D(2, padding='same')(x)

    x = layers.Conv2D(128, 7, activation='relu')(x)
    x = layers.BatchNormalization()(x)
    x = layers.MaxPooling2D(2, padding='same')(x)

    x = layers.Conv2D(256, 4, activation='relu')(x)
    x = layers.BatchNormalization()(x)
    x = layers.GlobalAveragePooling2D()(x)

    x = layers.Dropout(0.3)(x)
    out = layers.Dense(1024, name='embedding_vector')(x)
    return Model(inp, out, name='embedding')


@tf.keras.utils.register_keras_serializable(package='siamese')
class L1Dist(layers.Layer):
    # Element-wise absolute difference between two embeddings
    def call(self, inputs):
        a, b = inputs
        return tf.math.abs(a - b)


def make_siamese_model(embedding):
    img_a = layers.Input(shape=(IMG_SIZE, IMG_SIZE, 3), name='input_img')
    img_b = layers.Input(shape=(IMG_SIZE, IMG_SIZE, 3), name='validation_img')
    distance = L1Dist(name='l1_distance')([embedding(img_a), embedding(img_b)])
    score = layers.Dense(1, activation='sigmoid', name='similarity')(distance)
    return Model([img_a, img_b], score, name='SiameseNetwork')


embedding = make_embedding()
siamese_model = make_siamese_model(embedding)
siamese_model.summary()

## 6. Training

A custom training loop with `tf.GradientTape`:
forward pass → loss → gradients → apply. Validation runs every epoch;
the best weights (lowest val loss) are kept, and training stops early after `PATIENCE` epochs without improvement.

自己用 `GradientTape` 寫訓練迴圈，每個 epoch 跑一次驗證，保留 val loss 最低的權重，連續 `PATIENCE` 個 epoch 沒進步就提早停止。

In [ ]:
loss_fn = tf.keras.losses.BinaryCrossentropy()
optimizer = tf.keras.optimizers.Adam(LEARNING_RATE)
BEST_WEIGHTS = os.path.join(OUTPUT_DIR, 'best.weights.h5')

@tf.function
def train_step(x, y):
    with tf.GradientTape() as tape:
        y_hat = siamese_model(x, training=True)
        loss = loss_fn(y[:, None], y_hat)
    grads = tape.gradient(loss, siamese_model.trainable_variables)
    optimizer.apply_gradients(zip(grads, siamese_model.trainable_variables))
    return loss, y_hat

@tf.function
def eval_step(x, y):
    y_hat = siamese_model(x, training=False)
    return loss_fn(y[:, None], y_hat), y_hat

def run_epoch(ds, step_fn):
    loss_m, acc_m = tf.keras.metrics.Mean(), tf.keras.metrics.BinaryAccuracy()
    for x, y in ds:
        loss, y_hat = step_fn(x, y)
        loss_m.update_state(loss); acc_m.update_state(y[:, None], y_hat)
    return float(loss_m.result()), float(acc_m.result())

def train(epochs, patience):
    history = {k: [] for k in ('train_loss', 'train_acc', 'val_loss', 'val_acc')}
    best_val, wait = float('inf'), 0
    for epoch in range(1, epochs + 1):
        tr_loss, tr_acc = run_epoch(train_ds, train_step)
        va_loss, va_acc = run_epoch(val_ds, eval_step)
        for k, v in zip(history, (tr_loss, tr_acc, va_loss, va_acc)):
            history[k].append(v)
        msg = f'Epoch {epoch:3d} | train loss {tr_loss:.4f} acc {tr_acc:.3f} | val loss {va_loss:.4f} acc {va_acc:.3f}'
        if va_loss < best_val:
            best_val, wait = va_loss, 0
            siamese_model.save_weights(BEST_WEIGHTS)
            msg += '  ✔ best'
        else:
            wait += 1
        print(msg)
        if wait >= patience:
            print(f'Early stopping: no val improvement for {patience} epochs')
            break
    siamese_model.load_weights(BEST_WEIGHTS)
    return history

history = train(EPOCHS, PATIENCE)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for ax, metric in zip(axes, ('loss', 'acc')):
    ax.plot(history[f'train_{metric}'], label='train')
    ax.plot(history[f'val_{metric}'], label='val')
    ax.set_title(metric); ax.set_xlabel('epoch'); ax.legend()
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'training_curves.png'), dpi=120)
plt.show()

## 7. Evaluation

The decision threshold is chosen on the **validation** set, then applied once to the **test** set.
Results are reported overall and per subset:

* **self**: my anchor vs. my positive / LFW (the original task)
* **LFW**: unseen LFW people vs. each other (generalisation to new faces)

門檻值用驗證集決定，再套用到測試集。分別看「自己 vs 別人」和「沒看過的 LFW 人物互相比對」兩種結果。

In [ ]:
def predict(split):
    a, b, y, t = arrays[split]
    scores = siamese_model.predict(make_dataset(split, training=False), verbose=0).ravel()
    return scores, y, t

val_scores, val_y, _ = predict('val')
test_scores, test_y, test_t = predict('test')

# Pick the threshold that maximises accuracy on the validation set
candidates = np.unique(val_scores)
val_accs = [((val_scores >= th) == val_y).mean() for th in candidates]
THRESHOLD = float(candidates[int(np.argmax(val_accs))])
print(f'Threshold chosen on val: {THRESHOLD:.4f} (val acc {max(val_accs):.3f})')

def metrics(scores, y, th):
    pred = (scores >= th).astype(int)
    tn, fp, fn, tp = confusion_matrix(y, pred, labels=[0, 1]).ravel()
    return {
        'accuracy': (tp + tn) / len(y),
        'precision': tp / (tp + fp) if tp + fp else 0.0,
        'recall': tp / (tp + fn) if tp + fn else 0.0,
        'auc': roc_auc_score(y, scores) if len(set(y)) > 1 else float('nan'),
    }

subsets = {
    'overall': np.ones_like(test_t, dtype=bool),
    'self (me vs. others)': np.isin(test_t, [0, 1]),
    'LFW (unseen people)': np.isin(test_t, [2, 3]),
}
results = {name: metrics(test_scores[m], test_y[m], THRESHOLD) for name, m in subsets.items()}

# Markdown table you can paste into the README
print('\n| Test subset | Accuracy | Precision | Recall | ROC AUC |')
print('|---|---|---|---|---|')
for name, r in results.items():
    print(f"| {name} | {r['accuracy']:.3f} | {r['precision']:.3f} | {r['recall']:.3f} | {r['auc']:.3f} |")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
for name, m in subsets.items():
    fpr, tpr, _ = roc_curve(test_y[m], test_scores[m])
    axes[0].plot(fpr, tpr, label=f"{name} (AUC {results[name]['auc']:.3f})")
axes[0].plot([0, 1], [0, 1], 'k--', lw=0.8)
axes[0].set_xlabel('False positive rate'); axes[0].set_ylabel('True positive rate'); axes[0].set_title('ROC (test)')
axes[0].legend(loc='lower right')

for k, name in enumerate(PAIR_TYPES):
    axes[1].hist(test_scores[test_t == k], bins=30, alpha=0.6, label=name)
axes[1].axvline(THRESHOLD, color='k', ls='--', label='threshold')
axes[1].set_title('Score distribution by pair type (test)'); axes[1].set_xlabel('similarity score'); axes[1].legend()
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'evaluation.png'), dpi=120)
plt.show()

### 7.1 Error analysis
Look at the most confident mistakes. This is usually the fastest way to see what the model is really using.

看模型最有把握卻錯的例子，最能看出模型到底學到了什麼。

In [ ]:
a_idx, b_idx, _, _ = arrays['test']
errors = np.where((test_scores >= THRESHOLD) != test_y)[0]
errors = errors[np.argsort(-np.abs(test_scores[errors] - THRESHOLD))][:6]
print(f'{len(np.where((test_scores >= THRESHOLD) != test_y)[0])} errors out of {len(test_y)} test pairs')

if len(errors):
    fig, axes = plt.subplots(2, len(errors), figsize=(2.4 * len(errors), 5), squeeze=False)
    for k, i in enumerate(errors):
        axes[0, k].imshow(IMAGES[a_idx[i]]); axes[1, k].imshow(IMAGES[b_idx[i]])
        axes[0, k].set_title(f'{PAIR_TYPES[test_t[i]]}\nscore={test_scores[i]:.2f}', fontsize=9)
        axes[0, k].axis('off'); axes[1, k].axis('off')
    plt.tight_layout(); plt.show()

## 8. Save & reload

In [ ]:
MODEL_PATH = os.path.join(OUTPUT_DIR, 'siamese_model.keras')
siamese_model.save(MODEL_PATH)
with open(os.path.join(OUTPUT_DIR, 'results.json'), 'w') as f:
    json.dump({'threshold': THRESHOLD, 'test': results, 'epochs_trained': len(history['train_loss'])}, f, indent=2)

reloaded = tf.keras.models.load_model(MODEL_PATH, custom_objects={'L1Dist': L1Dist})
(xa, xb), _ = next(iter(test_ds))
assert np.allclose(reloaded.predict([xa, xb], verbose=0), siamese_model.predict([xa, xb], verbose=0), atol=1e-5)
print('Saved to', MODEL_PATH, '— reload check passed ✔')

## 9. Verification demo

To verify a new photo, compare it against several reference photos of the person and require that
enough of them match:

* **detection threshold** – score above which one comparison counts as a match (from section 7)
* **verification threshold** – fraction of reference images that must match

References are taken from the held-out **test** anchors, so they were never used in training.

驗證新照片時，拿它和多張參考照片比對，超過一定比例相符才算通過。參考照片取自測試集，模型訓練時沒看過。

In [ ]:
REFERENCE_PATHS = anc_split['test'][:: max(1, len(anc_split['test']) // 50)][:50]
REFERENCE_IMGS = tf.cast(tf.stack([load_image(p) for p in REFERENCE_PATHS]), tf.float32) / 255.0

def verify(model, input_img, detection_threshold=THRESHOLD, verification_threshold=0.5):
    # input_img: float32 tensor (IMG_SIZE, IMG_SIZE, 3) in [0, 1]
    batch = tf.repeat(input_img[None], len(REFERENCE_IMGS), axis=0)
    scores = model.predict([batch, REFERENCE_IMGS], verbose=0).ravel()
    match_ratio = float((scores >= detection_threshold).mean())
    return match_ratio >= verification_threshold, match_ratio, scores

# Example 1: a held-out photo of me (should verify)
me = tf.cast(load_image(pos_split['test'][-1]), tf.float32) / 255.0
print('held-out me     ->', verify(siamese_model, me)[:2])

# Example 2: an unseen LFW person (should NOT verify)
stranger_path = next(iter(lfw_split['test'].values()))[0]
stranger = tf.cast(load_image(stranger_path), tf.float32) / 255.0
print('unseen stranger ->', verify(siamese_model, stranger)[:2])

### 9.1 Live webcam check (Colab only)

In [ ]:
WEBCAM_DEMO = False  # set True to take a photo and verify it

if IN_COLAB and WEBCAM_DEMO:
    from IPython.display import Javascript, display
    from google.colab.output import eval_js
    from base64 import b64decode

    def take_photo(size=250):
        display(Javascript('''
        async function takePhoto(size) {
          const div = document.createElement('div'), btn = document.createElement('button');
          btn.textContent = 'Capture'; div.appendChild(btn);
          const video = document.createElement('video'); div.appendChild(video);
          const stream = await navigator.mediaDevices.getUserMedia({video: true});
          document.body.appendChild(div); video.srcObject = stream; await video.play();
          google.colab.output.setIframeHeight(document.documentElement.scrollHeight, true);
          await new Promise(r => btn.onclick = r);
          const s = Math.min(video.videoWidth, video.videoHeight), canvas = document.createElement('canvas');
          canvas.width = size; canvas.height = size;
          canvas.getContext('2d').drawImage(video, (video.videoWidth - s) / 2, (video.videoHeight - s) / 2, s, s, 0, 0, size, size);
          stream.getVideoTracks()[0].stop(); div.remove();
          return canvas.toDataURL('image/jpeg', 0.9);
        }'''))
        data = eval_js(f'takePhoto({size})')
        img = tf.io.decode_jpeg(b64decode(data.split(',')[1]), channels=3)
        return tf.image.resize(img, (IMG_SIZE, IMG_SIZE)) / 255.0

    photo = take_photo()
    verified, ratio, _ = verify(siamese_model, photo)
    plt.imshow(photo); plt.axis('off')
    plt.title(f"{'VERIFIED' if verified else 'NOT VERIFIED'}  (match ratio {ratio:.2f})")
    plt.show()

## 10. Limitations & next steps

* Only one "known" person; the model is a binary me / not-me verifier plus a small LFW verifier.
* No face detection / alignment: webcam images are a center crop, LFW images are pre-aligned. Adding a detector (e.g. OpenCV / MTCNN) would make the two sources more comparable.
* Small CNN trained from scratch. A pretrained backbone (e.g. MobileNetV2 / FaceNet embeddings) or a triplet / contrastive loss would likely generalise much better to unseen people.
* Evaluate on the official LFW *pairs* protocol for a number comparable with published work.